### Repo samples:

* Main PG repo: https://github.com/Azure-Samples/document-intelligence-code-samples
* Additional repo: https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/documentintelligence/azure-ai-documentintelligence/samples
* Previous Forms recognizer repo: https://github.com/microsoft/Form-Recognizer-Toolkit

## Import libraries:

In [30]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, ContentFormat,DocumentAnalysisFeature,AnalyzeOutputOption,AnalyzeResult
from azure.ai.documentintelligence.models import DocumentTable
from langchain.text_splitter import MarkdownHeaderTextSplitter
import os
import pandas as pd

from dotenv import load_dotenv
load_dotenv(override=True)

AZURE_DOC_INTELLIGENCE_ENDPOINT = os.environ["AZURE_DOC_INTELLIGENCE_ENDPOINT"]
AZURE_DOC_INTELLIGENCE_KEY = os.environ["AZURE_DOC_INTELLIGENCE_KEY"]

In [7]:
document_intelligence_client = DocumentIntelligenceClient(endpoint=AZURE_DOC_INTELLIGENCE_ENDPOINT, credential=AzureKeyCredential(AZURE_DOC_INTELLIGENCE_KEY), api_version="2024-11-30")

## Sample Image:

In [23]:
image_url = "https://github.com/Azure-Samples/document-intelligence-code-samples/blob/main/Data/layout/layout-finacialreport.png?raw=true"

poller = document_intelligence_client.begin_analyze_document("prebuilt-layout",
                                                             AnalyzeDocumentRequest(url_source=image_url),
                                                             output_content_format=ContentFormat.MARKDOWN)

result: AnalyzeResult = poller.result()
print(result.content)

# Beacon Roofing Supply, Inc. Notes to Consolidated Financial Statements Year Ended September 30, 2010 (dollars in thousands, except per share data or as otherwise indicated)


## 5. Prepaid Expenses and Other Current Assets

The significant components of prepaid expenses and other current assets were as follows:


<table>
<tr>
<th></th>
<th>September 30, 2010</th>
<th>September 30, 2009</th>
</tr>
<tr>
<td>Vendor rebates</td>
<td>$34.538</td>
<td>$38,857</td>
</tr>
<tr>
<td>Refundable income taxes</td>
<td>3,846</td>
<td>3,545</td>
</tr>
<tr>
<td>Other.</td>
<td>4.731</td>
<td>10,312</td>
</tr>
<tr>
<td></td>
<td>$43,115</td>
<td>$52,714</td>
</tr>
<tr>
<td>6. Property and Equipment, net</td>
<td></td>
<td></td>
</tr>
</table>


Property and equipment, net, consisted of the following:


<table>
<tr>
<th></th>
<th>September 30, 2010</th>
<th>September 30, 2009</th>
</tr>
<tr>
<td>Land</td>
<td>$ 3.056</td>
<td>$ 3,190</td>
</tr>
<tr>
<td>Buildings and leasehold improvements</td>
<td>19

### Page extraction Functions

In [9]:
def table_to_html(table):
    table_html = "<table>"
    rows = [
        sorted([cell for cell in table.cells if cell.row_index == i], key=lambda cell: cell.column_index)
        for i in range(table.row_count)
    ]
    for row_cells in rows:
        table_html += "<tr>"
        for cell in row_cells:
            tag = "th" if (cell.kind == "columnHeader" or cell.kind == "rowHeader") else "td"
            cell_spans = ""
            if cell.column_span is not None and cell.column_span > 1:
                cell_spans += f" colSpan={cell.column_span}"
            if cell.row_span is not None and cell.row_span > 1:
                cell_spans += f" rowSpan={cell.row_span}"
            table_html += f"<{tag}{cell_spans}>{html.escape(cell.content)}</{tag}>"
        table_html += "</tr>"
    table_html += "</table>"
    return table_html

def text_html_processing(OcrExtractionDIOutput):
    offset = 0
    page_map = []
    page_map_dict =[]

    for page_num, page in enumerate(OcrExtractionDIOutput.pages):
        tables_on_page = [
            table
            for table in (OcrExtractionDIOutput.tables or [])
            if table.bounding_regions and table.bounding_regions[0].page_number == page_num + 1
        ]
        #print(tables_on_page)

        # mark all positions of the table spans in the page
        page_offset = page.spans[0].offset
        page_length = page.spans[0].length
        table_chars = [-1] * page_length
        for table_id, table in enumerate(tables_on_page):
            for span in table.spans:
                # replace all table spans with "table_id" in table_chars array
                for i in range(span.length):
                    idx = span.offset - page_offset + i
                    if idx >= 0 and idx < page_length:
                        table_chars[idx] = table_id

        # build page text by replacing characters in table spans with table html
        page_text = ""
        added_tables = set()
        for idx, table_id in enumerate(table_chars):
            if table_id == -1:
                page_text += OcrExtractionDIOutput.content[page_offset + idx]
            elif table_id not in added_tables:
                page_text += table_to_html(tables_on_page[table_id])
                added_tables.add(table_id)

        page_text += " "
        page_map.append((page_num+1, offset, page_text))

        single_page_dict = {}
        single_page_dict['page_num']= page_num+1
        single_page_dict['content'] = page_text
        single_page_dict['offset'] = offset
        page_map_dict.append(single_page_dict)

        offset += len(page_text)

    return page_map_dict

### Page extraction Function

In [14]:
def OcrExtractionDI(relative_path: str, Markdown: [bool]=True):
    
    path_to_document = os.path.abspath(
        os.path.join(relative_path))
    
    if Markdown==True:
        output_format = ContentFormat.MARKDOWN
    else:
        output_format = None

    with open(path_to_document, "rb") as f:
        poller = document_intelligence_client.begin_analyze_document("prebuilt-layout", 
                                                                    analyze_request=f, content_type="application/octet-stream", 
                                                                    output_content_format=output_format)
    OcrExtractionDIOutput: AnalyzeResult = poller.result()    
    
    if Markdown==False:
        pagemap = text_html_processing(OcrExtractionDIOutput)
        extracted_processed_text = pagemap
    else:
        extracted_processed_text = OcrExtractionDIOutput

    return extracted_processed_text

In [24]:
Extracted_data =  OcrExtractionDI("../layout-pageobject.png",Markdown=True)
print(Extracted_data.content)

<!-- PageHeader="This is the header of the document." -->


# This is title


## 1. Text

Latin refers to an ancient Italic language
originating in the region of Latium in
ancient Rome.


## 2. Page Objects


### 2.1 Table

Here's a sample table below, designed to
be simple for easy understand and quick
reference.


<table>
<caption>Table 1: This is a dummy table</caption>
<tr>
<th>Name</th>
<th>Corp</th>
<th>Remark</th>
</tr>
<tr>
<td>Foo</td>
<td></td>
<td></td>
</tr>
<tr>
<td>Bar</td>
<td>Microsoft</td>
<td>Dummy</td>
</tr>
</table>


## 2.2. Figure


<figure>
<figcaption>Figure 1: Here is a figure with text</figcaption>

Values

500

450

100

400

350

300

300

250

200

200

200-

n

Jan

Feb

Mar

İçr

May

2um

Meness

</figure>


## 3. Others

Al Document Intelligence is an Al service
that applies advanced machine learning
to extract text, key-value pairs, tables,
and structures from documents
automatically and accurately:
☒
clear

☒
precise

☐
vague

☒
coherent

☐
Incomprehe

In [25]:
Extracted_data =  OcrExtractionDI("../10K-MSFT-07-27-2023.pdf",Markdown=True)
print(Extracted_data.content)

# UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K

☒
☒
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the Fiscal Year Ended June 30, 2023
OR

☐
☐
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the Transition Period From
to

Commission File Number 001-37845


# MICROSOFT CORPORATION


<table>
<tr>
<th>WASHINGTON</th>
<th>91-1144442</th>
</tr>
<tr>
<td>(STATE OF INCORPORATION)</td>
<td>(I.R.S. ID)</td>
</tr>
</table>


ONE MICROSOFT WAY, REDMOND, WASHINGTON 98052-6399
(425) 882-8080
www.microsoft.com/investor

Securities registered pursuant to Section 12(b) of the Act:


<table>
<tr>
<th>Title of each class</th>
<th>Trading Symbol</th>
<th>Name of exchange on which registered</th>
</tr>
<tr>
<td>Common stock, $0.00000625 par value per share</td>
<td>MSFT</td>
<td>NASDAQ</td>
</tr>
<tr>
<td>3.125% Notes due 2028</td>
<td>MSFT</td>
<td>NASDAQ</td>
</tr>
<tr>
<td>2.625% 

### Formatting output

In [35]:
def MdFormatting(ocr_extraction):
    doc_string = ocr_extraction.content
    ## Split the document into chunks base on markdown headers.
    headers_to_split_on = [
        ("#", "Title"),
        ("##", "Header 1"),
        ("###", "Header 2"),
    ]
    text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    markdown_chunks = text_splitter.split_text(doc_string)

    chunk_list = []
    for chunk in markdown_chunks:
        try:
            title = chunk.metadata['Title']
        except:
            title = ""
        try:
            header1 = chunk.metadata['Header 1']
        except:
            header1 = ""
        try:
            header2 = chunk.metadata['Header 2']
        except:
            header2 = ""

        chunk_list.append({"title": title,"header_1":header1,"header_2":header2,"content": title + "/n" + header1 + "/n"+ chunk.page_content})
    return pd.DataFrame(chunk_list)

In [37]:
output_df = MdFormatting(Extracted_data)

In [47]:
pd.set_option('display.max_rows', 500)

In [52]:
output_df[output_df['header_2']!=""]

,title,header_1,header_2,content
6,PART I,ITEM 1. BUSINESS,Embracing Our Future,PART I/nITEM 1. BUSINESS/nMicrosoft is a techn...
7,PART I,ITEM 1. BUSINESS,What We Offer,"PART I/nITEM 1. BUSINESS/nFounded in 1975, we ..."
8,PART I,ITEM 1. BUSINESS,The Ambitions That Drive Us,PART I/nITEM 1. BUSINESS/nTo achieve our visio...
9,PART I,ITEM 1. BUSINESS,Reinvent Productivity and Business Processes,"PART I/nITEM 1. BUSINESS/nAt Microsoft, we pro..."
10,PART I,ITEM 1. BUSINESS,Build the Intelligent Cloud and Intelligent Ed...,PART I/nITEM 1. BUSINESS/nAs digital transform...
11,PART I,ITEM 1. BUSINESS,Create More Personal Computing,PART I/nITEM 1. BUSINESS/nWe strive to make co...
12,PART I,ITEM 1. BUSINESS,Our Future Opportunity,PART I/nITEM 1. BUSINESS/nWe are focused on he...
13,PART I,ITEM 1. BUSINESS,Corporate Social Responsibility,PART I/nITEM 1. BUSINESS/n#### Commitment to S...
14,PART I,ITEM 1. BUSINESS,HUMAN CAPITAL RESOURCES,PART I/nITEM 1. BUSINESS/n#### Overview \nMic...
21,OPERATING SEGMENTS,Dynamics,Competition,OPERATING SEGMENTS/nDynamics/nCompetitors to O...


In [53]:
output_df

,title,header_1,header_2,content
0,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,,,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...
1,MICROSOFT CORPORATION,,,MICROSOFT CORPORATION/n/n<table>\n<tr>\n<th>WA...
2,MICROSOFT CORPORATION,DOCUMENTS INCORPORATED BY REFERENCE,,MICROSOFT CORPORATION/nDOCUMENTS INCORPORATED ...
3,MICROSOFT CORPORATION,MICROSOFT CORPORATION FORM 10-K For the Fiscal...,,MICROSOFT CORPORATION/nMICROSOFT CORPORATION F...
4,PARTI Item 1,Note About Forward-Looking Statements,,PARTI Item 1/nNote About Forward-Looking State...
5,PART I,ITEM 1. BUSINESS,,PART I/nITEM 1. BUSINESS/nGENERAL
6,PART I,ITEM 1. BUSINESS,Embracing Our Future,PART I/nITEM 1. BUSINESS/nMicrosoft is a techn...
7,PART I,ITEM 1. BUSINESS,What We Offer,"PART I/nITEM 1. BUSINESS/nFounded in 1975, we ..."
8,PART I,ITEM 1. BUSINESS,The Ambitions That Drive Us,PART I/nITEM 1. BUSINESS/nTo achieve our visio...
9,PART I,ITEM 1. BUSINESS,Reinvent Productivity and Business Processes,"PART I/nITEM 1. BUSINESS/nAt Microsoft, we pro..."
